# Qwen3-TTS LoRA — обучение голоса на русском
Три шага: установка, Google Drive, запуск интерфейса. Для Colab рекомендуется T4 GPU.


## 1. Установка
Эта ячейка получает свежий код ветки и ставит отдельные окружения для Qwen3-TTS и Qwen3-ASR.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

if not __import__('torch').cuda.is_available():
    raise RuntimeError('GPU не найден. В Colab выберите среду выполнения с T4 GPU.')
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ')

CODE = Path('/content/qwen3-tts-lora-code')
REPO = 'https://github.com/egor125552/audio-restoration-colab.git'
BRANCH = 'agent/qwen3-tts-lora-colab'
if CODE.exists():
    shutil.rmtree(CODE)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,str(CODE)], check=True)
subprocess.run(['bash', str(CODE/'qwen3_tts_lora_colab/install.sh'), '/content/qwen3-tts-trainer'], check=True)
print('Установка завершена.')


## 2. Google Drive
Checkpoint, готовые LoRA и кэш моделей будут сохраняться на Google Drive.


In [ ]:
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Qwen3-TTS Training')
ROOT.mkdir(parents=True, exist_ok=True)
HF = ROOT / '.cache' / 'huggingface'
HF.mkdir(parents=True, exist_ok=True)
os.environ['QWEN_TRAIN_DRIVE_ROOT'] = str(ROOT)
os.environ['HF_HOME'] = str(HF)
os.environ['HF_HUB_CACHE'] = str(HF / 'hub')
print(f'Проекты и checkpoint: {ROOT}')
print(f'Кэш моделей: {HF}')


## 3. Запуск интерфейса
Эта ячейка **должна работать постоянно**. Это не зависание. Вывод дочернего процесса специально возвращается через Python-ядро Colab, поэтому прогресс нарезки, ASR и обучения должен оставаться видимым ниже. Полоски прогресса сохраняют перерисовку на месте. Чтобы остановить сервер, прервите выполнение ячейки.


In [ ]:
import os, sys
from pathlib import Path

CODE = Path('/content/qwen3-tts-lora-code')
sys.path.insert(0, str(CODE / 'qwen3_tts_lora_colab'))
from colab_stream import run_streamed

os.environ['PYTHONUNBUFFERED'] = '1'
cmd = [
    '/content/qwen3-tts-trainer/tts-env/bin/python', '-u',
    str(CODE / 'qwen3_tts_lora_colab/launch_colab.py')
]
print('Запускаю интерфейс. Вывод ниже будет обновляться в реальном времени.', flush=True)
code = run_streamed(cmd)
if code != 0:
    raise RuntimeError(f'Интерфейс завершился с кодом {code}')
